# Carregando dados para teste

In [1]:
import pickle
from dateutil import parser as dateparser

In [2]:
DATA_PATH = "../data/science technology innovation.pkl"

In [3]:
with open(DATA_PATH, "rb") as f:
    data = pickle.load(f)

In [4]:
data[0]

{'title': "China celebrates National Science and Technology Workers' Day - China Daily",
 'description': "China celebrates National Science and Technology Workers' Day  China Daily",
 'published date': 'Sun, 31 May 2026 08:52:00 GMT',
 'url': 'https://news.google.com/rss/articles/CBMifkFVX3lxTE1idHZDUWt1ZHlhVEFfbU9rVWU0cGxPNjNxSVVzS0taRThHenR4RjcxU2h2SjBHX1Bmc0V3alVzUV9vTW1TZVFzSUI3UkZBaURxc2ZnV0RLcGp4cWJiaEVBalFKNVVtMmxSdG15RVZlbUpPbExyUlN2ZE9qb193UQ?oc=5&hl=en-US&gl=US&ceid=US:en',
 'publisher': {'href': 'https://www.chinadaily.com.cn',
  'title': 'China Daily'},
 'query': 'science technology innovation'}

# Preparando extrator

In [5]:
from vllm import LLM, SamplingParams
from transformers import AutoProcessor

In [ ]:
from glm_based_event_analysis.data.event_component_extraction import BaseEventComponentExtractor
from datetime import datetime

class vLLM(BaseEventComponentExtractor):

    def __init__(self, model_name: str, generation_params: dict):
        
        self._model = LLM(model_name)
        self._generation_params = generation_params
        self._processor = AutoProcessor.from_pretrained(model_name)
        self._system_prompt = self._get_system_prompt()
        
    def _create_system_user_prompts(self, title: str, 
                                          description: str, 
                                          url: str, 
                                          published_date: datetime) -> list[dict]:
        system_prompt = self._get_system_prompt()
        user_prompt_fn = self._get_user_prompt(title, description, url, published_date)

        input_pair = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt_fn}
        ]

        return input_pair
    
    def extract_event_components(self, titles: list[str],
                                       descriptions: list[str],
                                       urls: list[str],
                                       published_dates: list[datetime]) -> list[str]:
        
        # preparando os prompts
        messages_list = [
            self._create_system_user_prompts(
                title=title,
                description=description,
                url=url,
                published_date=published_date
            ) for title, description, url, published_date in zip(titles, descriptions, urls, published_dates)
        ]

        # aplicando o template de chat
        processed_texts = self._processor.apply_chat_template(
            messages_list,
            tokenize=False,
            add_generation_prompt=True,
        )

        # extraçao de componentes
        params = SamplingParams(**self._generation_params)
        outputs = self._model.generate(processed_texts, params, use_tqdm=True)

        # filtrando os outputs
        raw_components = []
        for output in outputs:
            generated_text = output.outputs[0].text
            raw_components.append(generated_text)
        
        # parsing
        parsed_components = []
        for raw_component in raw_components:
            parsed_component = self._parse_response(raw_component)
            if parsed_component is not None:
                parsed_components.append(parsed_component)

        return parsed_components
    

In [7]:
model_name = "Microsoft/Phi-4-mini-instruct"

In [8]:
extractor = vLLM(model_name=model_name)

INFO 06-10 16:12:40 [utils.py:240] non-default args: {'disable_log_stats': True, 'model': 'Microsoft/Phi-4-mini-instruct'}


[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


INFO 06-10 16:12:42 [model.py:568] Resolved architecture: Phi3ForCausalLM
INFO 06-10 16:12:42 [model.py:1697] Using max model len 4096
INFO 06-10 16:12:42 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 06-10 16:12:42 [vllm.py:886] Asynchronous scheduling is enabled.
INFO 06-10 16:12:42 [kernel.py:212] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
WARNING 06-10 16:12:47 [system_utils.py:157] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(EngineCore pid=50332) INFO 06-10 16:12:56 [core.py:109] Initializing a V1 LLM engine (v0.21.0) with config: model='Microsoft/Phi-4-mini-instruct', speculative_config=None, tokenizer='Microsoft/Phi-4-mini-instruct', skip_tokenizer_init=False, toke

(EngineCore pid=50332) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


(EngineCore pid=50332) INFO 06-10 16:13:01 [weight_utils.py:938] Filesystem type for checkpoints: EXT4. Checkpoint size: 7.15 GiB. Available RAM: 110.33 GiB.
(EngineCore pid=50332) INFO 06-10 16:13:01 [weight_utils.py:961] Auto-prefetch is disabled because the filesystem (EXT4) is not a recognized network FS (NFS/Lustre). If you want to force prefetching, start vLLM with --safetensors-load-strategy=prefetch.


...

(EngineCore pid=50332) INFO 06-10 16:13:02 [default_loader.py:397] Loading weights took 0.53 seconds
(EngineCore pid=50332) INFO 06-10 16:13:02 [gpu_model_runner.py:4959] Model loading took 7.17 GiB memory and 2.481817 seconds
(EngineCore pid=50332) INFO 06-10 16:13:06 [backends.py:1089] Using cache directory: /home/ksakiyama/.cache/vllm/torch_compile_cache/e492676786/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=50332) INFO 06-10 16:13:06 [backends.py:1148] Dynamo bytecode transform time: 3.51 s
(EngineCore pid=50332) INFO 06-10 16:13:07 [backends.py:292] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 0.440 s
(EngineCore pid=50332) INFO 06-10 16:13:07 [decorators.py:311] Directly load AOT compilation from path /home/ksakiyama/.cache/vllm/torch_compile_cache/torch_aot_compile/f1cd57d18593852ecbb2c6b7cd3ab3962b533f9bf92f208d43e01a18c2f9259e/rank_0_0/model
(EngineCore pid=50332) INFO 06-10 16:13:07 [monitor.py:53] torch.compile took 4.05

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:01<00:00, 34.19it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:00<00:00, 40.92it/s]


(EngineCore pid=50332) INFO 06-10 16:13:11 [gpu_model_runner.py:6243] Graph capturing finished in 3 secs, took 0.47 GiB
(EngineCore pid=50332) INFO 06-10 16:13:11 [gpu_worker.py:621] CUDA graph pool memory: 0.47 GiB (actual), 0.51 GiB (estimated), difference: 0.04 GiB (8.4%).
(EngineCore pid=50332) INFO 06-10 16:13:11 [jit_monitor.py:54] Kernel JIT monitor activated — Triton JIT compilations during inference will be logged as warnings.
(EngineCore pid=50332) INFO 06-10 16:13:11 [core.py:299] init engine (profile, create kv cache, warmup model) took 8.62 s (compilation: 4.05 s)


(EngineCore pid=50332) [transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


(EngineCore pid=50332) INFO 06-10 16:13:15 [vllm.py:886] Asynchronous scheduling is enabled.
(EngineCore pid=50332) INFO 06-10 16:13:15 [kernel.py:212] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])


In [9]:
titles = []
descriptions = []
urls = []
published_dates = []

for news in data[:500]:
    titles.append(news["title"])
    descriptions.append(news["description"])
    urls.append(news["url"])
    published_dates.append(dateparser.parse(news["published date"]))

In [10]:
teste = extractor.extract_event_components(
    titles=titles,
    descriptions=descriptions,
    urls=urls,
    published_dates=published_dates,
    temperature=0.0, max_tokens=512
)

Rendering prompts:   0%|          | 0/500 [00:00<?, ?it/s]

(EngineCore pid=50332) WARNING 06-10 16:13:19 [jit_monitor.py:103] Triton kernel JIT compilation during inference: _compute_slot_mapping_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.


Processed prompts:   0%|          | 0/500 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Error decoding JSON: Expecting ',' delimiter: line 12 column 22 (char 318)
Response was: {
  "keywords": "Viet Nam, digital, achievements, solutions",
  "what": "Award for best digital solutions in Viet Nam",
  "when": "2026-05-31",
  "where": [
    {
      "text": "National impact in Viet Nam",
      "locations": [
        {
          "country": "Vietnam",
          "lat": 21.0285,
          "long": 105.nx
        }
      ]
    }
  ],
  "who": "Nhan Dan Online",
  "why": "Recognizing impactful achievements in digital solutions",
  "how": "Award ceremony for best digital solutions"
}
Error decoding JSON: Expecting ',' delimiter: line 11 column 20 (char 341)
Response was: {
  "keywords": "middle-income trap, golden key, falling behind",
  "what": "Analysis of strategies to overcome the middle-income trap and avoid falling behind",
  "when": "2026-05-31",
  "where": {
    "text": "National impact on Vietnam",
    "locations": [
      {
        "country": "Vietnam",
        "lat": 21.0285

In [11]:
len(teste)

460